In [0]:
dbutils.widgets.dropdown(name='environment',defaultValue='dev',choices=['dev','qa','prod'],label='select Environment')

env=dbutils.widgets.get('environment')
print(env)

brnzTable=f'saleslake_{env}.bronze_{env}.rawstore'
print(brnzTable)
silverTable=f'saleslake_{env}.silver_{env}.cleanedstore'
print(silverTable)



In [0]:
from pyspark.sql import functions as F

df=spark.read.table(brnzTable)

int_lis=['store_id','region_id', 'square_feet']
strin_lis=['store_code','store_name','store_type','address','city','state','manager_name','status']
date_lis=['opening_date']
  
for i in int_lis:
  df=df.withColumn(i,F.col(i).cast('int'))
  
for i in strin_lis:
    df=df.withColumn(i,F.upper(F.trim(F.col(i))).cast('string')) 
  
for i in date_lis:
    df=df.withColumn(i,F.to_date(F.col(i)))
  
df=df.withColumn('ingest_ts',F.current_timestamp())

max_ingest_ts = spark.table(silverTable).agg(
    F.coalesce(F.max("ingest_ts"), F.to_timestamp(F.lit("1990-01-01"), "yyyy-MM-dd"))
).collect()[0][0]

df_filtered =df.filter(F.col("ingest_ts") > max_ingest_ts)
(df_filtered.write.format('delta').mode("append").saveAsTable(silverTable))

display(df_filtered)


  


In [0]:
%sql
select * from saleslake_qa.silver_qa.cleanedstore